# Device Comparison Analysis

Compare per-layer latency across 7 compute targets for Qwen2.5-1.5B Q4_K_M.

In [ ]:
import sys; sys.path.insert(0, '../..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from storage.parquet_store import ParquetStore

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10})

df = ParquetStore('../../data/parquet/all_7targets_qwen25_1.5b.parquet').read()
print(f'Total records: {len(df):,}')
print(f'Devices: {sorted(df["device"].unique())}')

## Sublayer Latency Breakdown

In [ ]:
MAIN = ['attention_qkv','flash_attention','attention_out','attn_norm',
       'ffn_gate','ffn_up','ffn_down','ffn_activation','ffn_norm',
       'rmsnorm','lm_head','embedding']
df_main = df[df['sublayer'].isin(MAIN)]

fig, ax = plt.subplots(figsize=(14,7))
pivot = df_main.groupby(['device','sublayer'])['latency_us'].sum().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
device_order = ['rpi4','orin_cpu','orin_gpu','op13_cpu','op13_gpu','op11_cpu','op11_gpu']
pivot_pct = pivot_pct.reindex(device_order)
pivot_pct.plot(kind='barh', stacked=True, ax=ax, colormap='tab20')
ax.set_xlabel('Latency Share (%)')
ax.set_title('Qwen2.5-1.5B Q4_K_M — Sublayer Latency Breakdown by Device')
ax.legend(bbox_to_anchor=(1.02,1), fontsize=8)
plt.tight_layout()
plt.savefig('../../claudedocs/figures/nb1_sublayer_breakdown.pdf', bbox_inches='tight')
plt.show()

## Per-Block Heatmap (CPU devices, decode phase)

In [ ]:
cpu_devices = ['rpi4','orin_cpu','op13_cpu','op11_cpu']
decode_cpu = df_main[(df_main['phase']=='decode') & (df_main['device'].isin(cpu_devices))]

fig, axes = plt.subplots(1,4, figsize=(20,8), sharey=True)
for i, device in enumerate(cpu_devices):
    dev = decode_cpu[decode_cpu['device']==device]
    pivot = dev.pivot_table(index='block_idx', columns='sublayer', values='latency_us', aggfunc='sum')
    pivot = pivot.reindex(range(28), fill_value=0)
    top_cols = pivot.sum().nlargest(6).index
    sns.heatmap(pivot[top_cols], ax=axes[i], cmap='YlOrRd', cbar=(i==3), yticklabels=(i==0))
    axes[i].set_title(device); axes[i].tick_params(axis='x', rotation=45, labelsize=7)
fig.suptitle('Decode Phase — Per-Block Latency (CPU)', y=1.02)
plt.tight_layout()
plt.savefig('../../claudedocs/figures/nb1_block_heatmap.pdf', bbox_inches='tight')
plt.show()

## Performance Summary

In [ ]:
for device in device_order:
    dev = df_main[df_main['device']==device]
    total = dev['latency_us'].sum()
    top = dev.groupby('sublayer')['latency_us'].sum().nlargest(3)
    items = ', '.join([f'{n}({v/total*100:.0f}%)' for n,v in top.items()])
    print(f'{device:12s}: total={total/1e6:.2f}s  top: {items}')